# 1. Three body problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class TBP:
    def __init__(self, m, pos, vel):
        """
        三体问题的类
        m: 质量 [m0, m1, m2]
        pos: 位置 [[x0,y0], [x1,y1], [x2,y2]]
        vel: 速度 [[vx0,vy0], [vx1,vy1], [vx2,vy2]]
        """
        self.m = np.array(m)
        self.n = len(m)
        
        # 初始状态: [x0,y0,x1,y1,x2,y2,vx0,vy0,vx1,vy1,vx2,vy2]
        self.y0 = np.zeros(12)
        for i in range(3):
            self.y0[2*i:2*i+2] = pos[i]
            self.y0[6+2*i:6+2*i+2] = vel[i]
    
    def g_acc(self, r1, r2, m_val):
        """
        计算引力加速度
        r1: 位置1
        r2: 位置2  
        m_val: 质量
        """
        r_v = r2 - r1
        r_m = np.linalg.norm(r_v)
        
        if r_m < 1e-10:  # 避免除零导致报错
            return np.zeros(2)
        
        # 加速度 
        a_m = m_val / (r_m**3)
        return a_m * r_v
    
    def deriv(self, t, y):
        """
        计算导数
        y: 状态向量
        返回: 导数向量
        """
        dydt = np.zeros(12)
        
        # 位置导数 = 速度
        dydt[0:6] = y[6:12]
        
        # 速度导数 = 加速度
        p = [y[0:2], y[2:4], y[4:6]]
        
        for i in range(3):
            a = np.zeros(2)
            for j in range(3):
                if i != j:
                    # j对i的引力
                    acc = self.g_acc(p[i], p[j], self.m[j])
                    a += acc
            
            dydt[6+2*i:6+2*i+2] = a
        
        return dydt
    
    def rk4(self, dt, T):
        """
        4阶Runge-Kutta方法
        dt: 时间步长
        T: 总时间
        """
        n_steps = int(T / dt)
        t_arr = np.linspace(0, T, n_steps + 1)
        
        # 把解存入数组
        sol = np.zeros((n_steps + 1, 12))
        sol[0] = self.y0
        
        for i in range(n_steps):
            t_curr = t_arr[i]
            y_curr = sol[i]
            k1 = dt * self.deriv(t_curr, y_curr)
            k2 = dt * self.deriv(t_curr + dt/2, y_curr + k1/2)
            k3 = dt * self.deriv(t_curr + dt/2, y_curr + k2/2)
            k4 = dt * self.deriv(t_curr + dt, y_curr + k3)
            y_next = y_curr + (k1 + 2*k2 + 2*k3 + k4) / 6
            sol[i+1] = y_next
        
        return t_arr, sol
    
    def get_traj(self, sol):
        """
        提取出轨迹
        sol: 解的数组
        """
        traj = []
        for i in range(3):
            x_t = sol[:, 2*i]
            y_t = sol[:, 2*i+1]
            traj.append((x_t, y_t))
        return traj
    
    def chk_col(self, pos, tol=1e-3):
        """
        检查共线性
        pos: 三个位置
        tol: 小于它认为共线
        """
        p0, p1, p2 = pos
        
        # 计算三角形面积
        area = 0.5 * abs(
            p0[0]*(p1[1]-p2[1]) + 
            p1[0]*(p2[1]-p0[1]) + 
            p2[0]*(p0[1]-p1[1])
        )
        
        return area < tol, area

In [ ]:
def plt_orb(traj, m, tt, fn=None):
    """
    画轨道图
    """
    plt.figure(figsize=(10, 8))
    
    clr = ['red', 'blue', 'green']
    lbl = [f'B{i} (m={m[i]})' for i in range(3)]
    
    for i, (x_t, y_t) in enumerate(traj):
        plt.plot(x_t, y_t, color=clr[i], label=lbl[i], linewidth=2)
        # 起点
        plt.scatter(x_t[0], y_t[0], color=clr[i], s=100, marker='o', edgecolors='black')
        # 终点
        plt.scatter(x_t[-1], y_t[-1], color=clr[i], s=100, marker='s', edgecolors='black')
    
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title(tt)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    
    if fn:
        plt.savefig(fn, dpi=300, bbox_inches='tight')
    
    plt.show()

## a. Euler’s stable solution 

In [ ]:
print("Part (a): Euler Stable Solution")
print("-" * 40)
m_a = [3.0, 1.0, 1.0]
pos_a = [[0.0, 1.0], [0.0, -0.83299], [0.0, -2.16700]]
vel_a = [[-0.63034, 0.0], [0.52507, 0.0], [1.36596, 0.0]]
dt = 0.01
T_a = 7.0

# 把类实力化
pb_a = TBP(m_a, pos_a, vel_a)
print("Solving with RK4...")
t_a, sol_a = pb_a.rk4(dt, T_a)

# 提取出轨迹
traj_a = pb_a.get_traj(sol_a)

plt_orb(traj_a, m_a, "Three-body orbits, Euler's solution", "fig_a.png")

## b.collinear or not

In [ ]:
print("Part (b): Check Collinearity")
print("-" * 50)

t_chk = [1.0, 2.0, 4.0, 6.0]

for tc in t_chk:
    # 找最近的时间点
    idx = np.argmin(np.abs(t_a - tc))
    
    # 提取位置
    pos_t = []
    for i in range(3):
        x = sol_a[idx, 2*i]
        y = sol_a[idx, 2*i+1]
        pos_t.append([x, y])
    
    # 检查是否共线
    is_col, area = pb_a.chk_col(pos_t)
    
    print(f"T = {tc}:")
    print(f"  Pos: B0={pos_t[0]}, B1={pos_t[1]}, B2={pos_t[2]}")
    print(f"  Area = {area:.6e}")
    print(f"  Collinear: {is_col}")
    print()

## c. Chaotic Solution

In [ ]:
print("Part (c): Chaotic Solution")
print("-" * 40)

m_c = [1.0, 1.0, 1.0]
pos_c = [[3.0, 0.0], [-1.5, 2.59], [-0.6, -2.65]]
vel_c = [[0.1, 0.2], [-0.2, -0.1], [0.1, -0.1]]
T_c = 20.0

pb_c = TBP(m_c, pos_c, vel_c)
print("Solving with RK4...")
t_c, sol_c = pb_c.rk4(dt, T_c)

traj_c = pb_c.get_traj(sol_c)
plt_orb(traj_c, m_c, "Chaotic Solution - TBP", "fig_c.png")

print("\nDone!")
print("Plots: fig_a.png, fig_c.png")